# 🧭 Трекинг объектов: KLT и Deep SORT (Colab)

**Цель**: понять различия KLT и Deep SORT, запустить оба треккера на видео и сравнить устойчивость.

**Содержание:**
1. Теория (коротко): KLT vs Deep SORT  
2. Установка зависимостей (в Colab)  
3. Генерация тестового видео (без внешних файлов)  
4. Демонстрация KLT (оптический поток + переразметка фич)  
5. Deep SORT + YOLOv8 (детекция + ассоциация треков)  
6. Мини-оценка и задания (YOUR CODE HERE)  

> 💡 Все клетки можно запускать последовательно сверху вниз.


## 1) Теория (кратко)

- **KLT** (Kanade–Lucas–Tomasi): отслеживает **точки интереса** и их смещения между кадрами с помощью оптического потока (PyrLK). Быстрый, но плохо переносит перекрытия и не восстанавливает потерянные треки.
- **Deep SORT**: использует **детектор объектов** (например, YOLOv8) + **Kalman Filter** + **Hungarian Algorithm** и **Re-ID эмбеддинги** для устойчивой ассоциации ID между кадрами. Лучше работает при множестве объектов и перекрытиях.


## 2) Установка зависимостей (для Colab)

При запуске **в Google Colab** выполните ячейку ниже. В локальной среде пропустите или адаптируйте.


In [ ]:
# Если вы в Google Colab, раскомментируйте:
# !pip -q install opencv-python opencv-contrib-python ultralytics deep-sort-realtime filterpy lapx tqdm

import sys, cv2, numpy as np, os, time
print("OpenCV:", cv2.__version__)


## 3) Генерация тестового видео

Создадим синтетическое видео `synthetic_demo.mp4` с несколькими движущимися объектами.


In [ ]:
import cv2, numpy as np, math, random

w, h = 640, 360
fps = 25
seconds = 12
num_objs = 6

# Инициализация случайных объектов (круги/прямоугольники)
objs = []
for i in range(num_objs):
    x = random.randint(50, w-50)
    y = random.randint(50, h-50)
    vx = random.choice([-3, -2, -1, 1, 2, 3])
    vy = random.choice([-3, -2, -1, 1, 2, 3])
    shape = random.choice(["circle", "rect"])
    size = random.randint(12, 24)
    objs.append([x, y, vx, vy, shape, size])

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('synthetic_demo.mp4', fourcc, fps, (w, h))

for t in range(fps*seconds):
    frame = np.zeros((h, w, 3), dtype=np.uint8)
    # фон - слабый шум
    noise = np.random.randint(0, 10, (h,w,1), dtype=np.uint8)
    frame[:,:,0] = noise[:,:,0]
    frame[:,:,1] = noise[:,:,0]
    frame[:,:,2] = noise[:,:,0]

    for i in range(num_objs):
        x, y, vx, vy, shape, size = objs[i]
        # Отскоки от границ
        if x < size or x > w-size: vx *= -1
        if y < size or y > h-size: vy *= -1
        x += vx; y += vy
        objs[i][0], objs[i][1], objs[i][2], objs[i][3] = x, y, vx, vy

        if shape == "circle":
            cv2.circle(frame, (int(x), int(y)), size, (50+20*i, 180-20*i, 120+10*i), -1)
        else:
            cv2.rectangle(frame, (int(x-size), int(y-size)), (int(x+size), int(y+size)), (150-20*i, 50+20*i, 200-10*i), -1)

    out.write(frame)

out.release()
print("Saved synthetic video to synthetic_demo.mp4")


## 4) KLT: трекинг признаков (PyrLK)

Подход:  
1) На первом кадре ищем **хорошие точки** (`cv2.goodFeaturesToTrack`).  
2) Отслеживаем их между кадрами (`cv2.calcOpticalFlowPyrLK`).  
3) Периодически **добавляем новые точки** (re-seed), чтобы компенсировать потерю.

Ниже — минимальная реализация с визуализацией треков.


In [ ]:
import cv2, numpy as np

video_path = 'synthetic_demo.mp4'
cap = cv2.VideoCapture(video_path)

feature_params = dict(maxCorners=300, qualityLevel=0.2, minDistance=7, blockSize=7)
lk_params = dict(winSize=(21,21), maxLevel=3,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 20, 0.03))

# Инициализация
ret, old_frame = cap.read()
if not ret:
    raise RuntimeError("Не удалось прочитать видео")
old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)

mask = np.zeros_like(old_frame)  # для рисования траекторий

# настройки re-seed
reseed_interval = 10  # каждые N кадров добавляем новые точки
frame_idx = 0

# Запись результата
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('klt_tracks.mp4', fourcc, 25, (old_frame.shape[1], old_frame.shape[0]))

while True:
    ret, frame = cap.read()
    if not ret: break
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Оптический поток
    p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)
    if p1 is None:
        # если все потеряли — пересемплируем
        p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
        old_gray = frame_gray.copy()
        mask = np.zeros_like(mask)
        frame_idx += 1
        continue

    good_new = p1[st==1]
    good_old = p0[st==1]

    # Рисуем треки
    for (new, old) in zip(good_new, good_old):
        a,b = new.ravel().astype(int)
        c,d = old.ravel().astype(int)
        cv2.line(mask, (a,b), (c,d), (0,255,0), 1)
        cv2.circle(frame, (a,b), 2, (0,0,255), -1)

    img = cv2.add(frame, mask)
    out.write(img)

    # Подготовка к следующей итерации
    old_gray = frame_gray.copy()
    p0 = good_new.reshape(-1,1,2)

    # Периодический re-seed
    frame_idx += 1
    if frame_idx % reseed_interval == 0:
        more = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
        if more is not None:
            p0 = np.concatenate([p0, more], axis=0)

cap.release()
out.release()
print("Saved KLT visualization to klt_tracks.mp4")


## 5) Deep SORT + YOLOv8

**Пайплайн**: детектор (YOLOv8n) → трекер (Kalman + Hungarian + Re-ID из `deep-sort-realtime`).

> ⚠️ В этой среде интернет отключён, поэтому установка пакетов не выполняется.
> Запустите эту ячейку **в Colab** после установки зависимостей.


In [ ]:
# Этот код ожидает, что вы уже выполнили:
# !pip -q install ultralytics deep-sort-realtime filterpy lapx tqdm

import sys, os, time
import numpy as np

try:
    from ultralytics import YOLO
    from deep_sort_realtime.deepsort_tracker import DeepSort
    import cv2
except Exception as e:
    print("Похоже, пакеты не установлены. Запустите в Colab и выполните установку.", e)

def run_deepsort_yolo(video_in='synthetic_demo.mp4', out_path='deepsort_yolo.mp4',
                      conf=0.25, classes=None):
    '''
    classes: список ID классов COCO (например, [0] — только 'person'),
             или None, чтобы оставить все классы.
    '''
    model = YOLO('yolov8n.pt')
    tracker = DeepSort(max_age=30, n_init=3, nms_max_overlap=1.0, max_cosine_distance=0.4)

    cap = cv2.VideoCapture(video_in)
    if not cap.isOpened():
        raise RuntimeError("Не удалось открыть видео")

    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, fps, (w, h))
    palette = {}

    def color_for_id(idx):
        import random
        random.seed(idx)
        return (random.randint(60,255), random.randint(60,255), random.randint(60,255))

    while True:
        ret, frame = cap.read()
        if not ret: break

        # Детекция
        results = model.predict(frame, conf=conf, verbose=False)[0]
        dets = []
        for b in results.boxes:
            cls_id = int(b.cls.item())
            if (classes is not None) and (cls_id not in classes):
                continue
            x1, y1, x2, y2 = map(int, b.xyxy[0].tolist())
            score = float(b.conf.item())
            dets.append(([x1, y1, x2-x1, y2-y1], score, cls_id))

        # Ассоциация Deep SORT
        tracks = tracker.update_tracks(dets, frame=frame)

        # Рисуем
        for tr in tracks:
            if not tr.is_confirmed() or tr.time_since_update > 0:
                continue
            ltrb = tr.to_ltrb()
            x1, y1, x2, y2 = map(int, ltrb)
            tid = tr.track_id
            if tid not in palette:
                palette[tid] = color_for_id(tid)
            color = palette[tid]
            cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
            label = f"ID {tid}"
            cv2.putText(frame, label, (x1, y1-6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        out.write(frame)

    cap.release()
    out.release()
    print(f"Saved Deep SORT result to {out_path}")

# Пример запуска (после установки зависимостей в Colab):
# run_deepsort_yolo('synthetic_demo.mp4', 'deepsort_yolo.mp4', conf=0.25, classes=None)


## 6) Мини-оценка и задания

Ниже — утилита для **очень простой** оценки непрерывности трека (прокси-метрика):
- Считаем, сколько кадров подряд точка (KLT) или объект (Deep SORT) удерживает ID без пропусков.
- Вам предлагается **дописать фрагмент** кода (YOUR CODE HERE).


In [ ]:
from collections import defaultdict
import json

def continuity_metric(id_timeline):
    """Подсчет средней длины непрерывных сегментов по ID.
    id_timeline: список ID по кадрам (например, [None, 1,1,1, None, 2,2, 1,1])
    Возвращает среднюю длину сегмента и распределение сегментов.
    """
    segs = []
    cur_id, cur_len = None, 0
    for tid in id_timeline:
        if tid is None:
            if cur_len > 0:
                segs.append(cur_len)
            cur_id, cur_len = None, 0
            continue
        if cur_id is None or tid != cur_id:
            if cur_len > 0:
                segs.append(cur_len)
            cur_id, cur_len = tid, 1
        else:
            cur_len += 1
    if cur_len > 0:
        segs.append(cur_len)
    if not segs:
        return 0.0, []
    return float(sum(segs))/len(segs), segs

# === ЗАДАНИЕ 1 (KLT):
# Реализуйте извлечение таймлайна трека для ОДНОЙ самой "долгой" точки KLT.
# Подсказка: можно модифицировать KLT-цикл, чтобы сохранять индексы/координаты точек по кадрам.
# Сюда можно подать список ID/индексов точки (или None при потере).
klt_id_timeline = []  # YOUR CODE HERE

avg_len, segs = continuity_metric(klt_id_timeline)
print("KLT continuity avg_len=", avg_len, "segments:", segs)

# === ЗАДАНИЕ 2 (Deep SORT):
# Сохраните ID активных треков по кадрам (например, самый "старый" подтвержденный).
# Заполните ds_id_timeline и посчитайте метрику.
ds_id_timeline = []  # YOUR CODE HERE

avg_len_ds, segs_ds = continuity_metric(ds_id_timeline)
print("Deep SORT continuity avg_len=", avg_len_ds, "segments:", segs_ds)

# === ЗАДАНИЕ 3:
# Попробуйте добавить (или уменьшить) шум/размер объектов в синтетическом видео и сравнить метрику.


### Экспорт результатов

Сохраните полученные видео `klt_tracks.mp4` и (в Colab) `deepsort_yolo.mp4`.
Можно скачать их на локальный компьютер через файловый браузер Colab.


### Бонус: Сглаживание KLT трека фильтром Калмана (опционально)

**Идея:** применить Калмана к координатам точки для подавления дрожания.


In [ ]:
# Пример каркаса (без полной реализации) фильтра Калмана для точки:
# from filterpy.kalman import KalmanFilter
# kf = KalmanFilter(dim_x=4, dim_z=2)
# kf.F = np.array([[1,0,1,0],
#                  [0,1,0,1],
#                  [0,0,1,0],
#                  [0,0,0,1]], dtype=float)
# kf.H = np.array([[1,0,0,0],
#                  [0,1,0,0]], dtype=float)
# kf.P *= 50.0
# kf.R *= 1.0
# kf.Q = np.eye(4)*0.01
# # Для каждого наблюдения (x,y):
# # kf.predict(); kf.update(np.array([x,y]))
# # filtered_state = kf.x.copy()
pass


---

## Готово ✅

В этом ноутбуке:
- KLT-трекинг на синтетическом видео,
- каркас Deep SORT + YOLOv8 (запускается в Colab после установки пакетов),
- простая метрика непрерывности,
- задания **YOUR CODE HERE** для закрепления.

Удачи! 💪
